Step 3 Data Visualization

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
import branca.colormap as cm

In [ ]:
def render_final_map(geojson_file, csv_file):
    """
    Reads the output files and renders the final map, highlighting the 
    Balanced Route as the primary recommendation.
    """
    gdf = gpd.read_file(geojson_file)
    summary_df = pd.read_csv(csv_file)
    
    # Determine the initial map center
    first_geom = gdf.iloc[0].geometry
    start_point = first_geom.coords[0] 
    map_center = [start_point[1], start_point[0]] 
    
    # Initialize the base map (Removed fixed zoom_start to allow auto-fitting later)
    m = folium.Map(location=map_center, tiles="cartodbdark_matter")
    
    # Risk color scale 
    risk_col = 'risk_score'
    if risk_col in gdf.columns:
        risk_cmap = cm.LinearColormap(
            colors=['#00FF00', '#ADFF2F', '#FFFF00', '#FFA500', '#FF0000'], 
            vmin=0.0, vmax=1.0, caption="Risk Level", text_color='white'
        )
        m.add_child(risk_cmap)
        get_color = lambda x: risk_cmap(x)

    else:
        print(f"Warning: Column '{risk_col}' not found in GeoJSON!")
        get_color = lambda x: "#39FF14" 

    # Use FeatureGroup to group routes for LayerControl
    route_groups = {}

    for idx, row in gdf.iterrows():
        route_type = str(row.get('route_type', '')).lower()
        route_name = str(row.get('route_type', f'Route {idx}')).title()
        
        # If this route group doesn't exist yet, create a new FeatureGroup
        if route_name not in route_groups:
            route_groups[route_name] = folium.FeatureGroup(name=route_name)
            m.add_child(route_groups[route_name])
            
        dash_style = "10, 10"
        line_weight = 5        
        line_opacity = 0.6     

        if "balanced" in route_type:
            dash_style = None  
            line_weight = 5
            line_opacity = 1.0 
            
        tooltip_html = "<br>".join([f"<b>{k}:</b> {v}" for k, v in row.drop('geometry').items()])
            
        # Draw this line on the specific FeatureGroup
        folium.GeoJson(
            row.geometry,
            style_function=lambda f, r_val=row.get(risk_col, 0), d=dash_style, w=line_weight, o=line_opacity: {
                "color": get_color(r_val),
                "weight": w,
                "opacity": o,
                "dashArray": d
            },
            tooltip=folium.Tooltip(tooltip_html)
        ).add_to(route_groups[route_name])


    # Prioritize the 'balanced' route for extracting start/end points; fallback to entire dataset
    balanced_gdf = gdf[gdf['route_type'].str.contains('balanced', case=False, na=False)]
    target_gdf = balanced_gdf if not balanced_gdf.empty else gdf

    if not target_gdf.empty:
        # First point of the first segment
        real_start = target_gdf.iloc[0].geometry.coords[0]
        # Last point of the last segment
        real_end = target_gdf.iloc[-1].geometry.coords[-1]

        # Add Start Logo
        folium.Marker(
            location=[real_start[1], real_start[0]], # Note: Folium requires [latitude, longitude]
            icon=folium.Icon(color="green", icon="play")
        ).add_to(m)

        # Add End Logo
        folium.Marker(
            location=[real_end[1], real_end[0]],
            icon=folium.Icon(color="red", icon="stop")
        ).add_to(m)

    # Fit map bounds to encompass all routes
    bounds = gdf.total_bounds  # Returns format: [min_lon, min_lat, max_lon, max_lat]
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

    # Add layer control and save
    folium.LayerControl().add_to(m)
    m.save("recommended_route_map.html")
    print("Successfully generated map: recommended_route_map.html")
    
    return m

In [16]:

# Execute code
if __name__ == "__main__":
    render_final_map("step2_current_routes.geojson", "step2_route_summary.csv")

Loading Map Data from step2_current_routes.geojson...
Loading Summary Data from step2_route_summary.csv...
Successfully generated map: recommended_route_map.html
